# 04 · Variant Validation & Robustness Evaluation (Stage 4 — Experiments E2 & E3)
This notebook **does not generate variants**. It reads the variant CSV files your team placed in `data/variants/`, validates them (Section 8.4), splits the LLM variants into a training pool and a held-out test set (Section 8.5), and measures how the E1 baseline models perform on them.

| Part | What it does |
|---|---|
| A · Input contract | Expected CSV format; writes a header-only template file |
| B · Validation (8.4) | 5 checks: (1) format/numeric, (2) range clipping to training bounds, (3) logical consistency, (4) fixed-feature integrity, (5) distribution similarity + copy/duplicate removal. Discard counts and rates are logged |
| C · Split | LLM variants → `variant_train_pool` / `variant_test_heldout` (group-aware when `source_row_id` is given); scripted variants → E3 control set |
| D · E2 / E3 | All 4 baseline models × 3 seeds on the held-out LLM variants (E2) and the scripted control variants (E3) |
| E · Metrics | **Recall Drop** = original recall − variant recall, **Evasion Rate** = share of attack variants predicted `Benign`, per model and attack type |

**Prerequisites:** notebooks 01 and 02 have been run. A CPU runtime is enough (a GPU only speeds up LSTM inference).

## 1 · Mount Drive, load configuration and the shared module

In [ ]:
import os, sys, json, glob, time, gc, re
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/IDS_Polymorphic_Project'
else:
    BASE_DIR = os.path.abspath('./IDS_Polymorphic_Project')

PCFG = json.load(open(os.path.join(BASE_DIR, 'config', 'project_config.json')))
FCFG = json.load(open(os.path.join(BASE_DIR, 'config', 'feature_config.json')))
DIRS = {k: v.replace(PCFG['dirs']['base'], BASE_DIR) for k, v in PCFG['dirs'].items()}
DIRS['code'] = os.path.join(BASE_DIR, 'code')
DIRS['baseline_models'] = os.path.join(DIRS['models'], 'baseline')
DIRS['validated'] = os.path.join(DIRS['variants'], 'validated')
DIRS['runs04'] = os.path.join(DIRS['results'], '04_runs')
for d in DIRS.values():
    os.makedirs(d, exist_ok=True)

sys.path.insert(0, DIRS['code'])
import importlib, ids_common
importlib.reload(ids_common)
from ids_common import *

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib, torch
from scipy.stats import ks_2samp
from sklearn.model_selection import train_test_split, GroupShuffleSplit

CLASS_NAMES = PCFG['class_names']; BENIGN = PCFG['benign_class']; BENIGN_ID = PCFG['benign_id']
ATTACK_CLASSES = PCFG['attack_classes']; LABEL_TO_ID = PCFG['label_to_id']
MODEL_SEEDS = PCFG['model_seeds']; DATA_SEED = PCFG['data_seed']
FEATURES = FCFG['feature_names']; MUTABLE = FCFG['mutable_features']; FIXED = FCFG['fixed_features']
GBOUNDS = FCFG['feature_bounds_train']['global']
DEVICE = get_device()

train_raw = pd.read_parquet(os.path.join(DIRS['processed'], 'train_raw.parquet'))
scaler = joblib.load(os.path.join(DIRS['processed'], 'scaler.joblib'))
print('device:', DEVICE, '| features:', len(FEATURES), f'(mutable {len(MUTABLE)}, fixed {len(FIXED)})')
print('attack classes:', ATTACK_CLASSES)

## 2 · Part A — Input contract for `data/variants/`
Put one or more **`.csv`** files in `MyDrive/IDS_Polymorphic_Project/data/variants/` (any file names; files starting with `TEMPLATE` are ignored).

| Column | Required | Meaning |
|---|---|---|
| every name in `config/feature_config.json → feature_names` | yes | raw (unscaled) feature values, same units as `data/processed/train_raw.parquet` |
| `Label` | yes | attack class: one of the attack classes above (raw CIC names such as `FTP-Patator` are mapped too) |
| `source` | recommended | `llm` (E2 / E4) or `scripted` (E3 control). If missing, files whose name contains `scripted`, `control` or `random` count as scripted, all others as `llm` |
| `source_row_id` | recommended | 0-based row position in `train_raw.parquet` of the real flow the variant was derived from. Enables the exact fixed-feature check and a leakage-free group split |
| `style` | optional | variation style tag (e.g. `slower`, `bursty`, `smaller_packets`, `mixed`); used for per-style breakdowns |
| `generator` | optional | model / prompt version, kept for reproducibility |

Extra columns are kept as metadata and never used as model features.

In [ ]:
META_COLS = ['Label', 'source', 'source_row_id', 'style', 'generator']
template_path = os.path.join(DIRS['variants'], 'TEMPLATE_variant_format.csv')
pd.DataFrame(columns=FEATURES + META_COLS).to_csv(template_path, index=False)
print('Header-only template written to', template_path)

variant_files = sorted(f for f in glob.glob(os.path.join(DIRS['variants'], '*.csv'))
                       if not os.path.basename(f).upper().startswith('TEMPLATE'))
print(f'\nFound {len(variant_files)} variant file(s):')
for f in variant_files:
    print(f'  {os.path.basename(f)}  ({os.path.getsize(f) / 1024:.1f} KB)')
if not variant_files:
    raise FileNotFoundError(f"No variant CSV files in {DIRS['variants']}. Add files following the contract above, then re-run.")

## 3 · Validation settings

In [ ]:
RANGE_MODE        = 'clip'   # 'clip' = clip out-of-range values to train [min, max] (plan 8.4); 'discard' = drop such rows
RATE_TOLERANCE    = 1.5      # allowed ratio between a reported rate and the rate implied by counts/bytes and duration
RULE_MIN_REAL_PASS = 0.98    # a consistency rule is used only if >= 98 % of REAL training attack flows satisfy it
DURATION_UNIT_PER_SECOND = 1e6   # CIC-IDS2017 Flow Duration / IATs are in microseconds
FIXED_RTOL        = 1e-5
HELDOUT_FRACTION  = 0.40     # share of validated LLM variants kept as the never-trained-on test set
KS_SUBSAMPLE      = 5000

## 4 · Part B — Load files and run check (1): format / numeric
Rows are discarded if a feature column is missing, any feature is non-numeric / NaN / Inf, or the label is not one of the attack classes.

In [ ]:
label_lookup = {c.upper(): c for c in ATTACK_CLASSES}
label_lookup.update({k.strip().upper(): v for k, v in PCFG['label_map'].items() if v in ATTACK_CLASSES})

def infer_source(fname):
    return 'scripted' if re.search(r'(scripted|control|random)', fname, re.I) else 'llm'

discard_log = []          # one entry per discarded row: file, row, source, Label, check, reason
frames = []
for f in variant_files:
    raw = pd.read_csv(f, low_memory=False)
    raw.columns = raw.columns.str.strip()
    fname = os.path.basename(f)
    missing = [c for c in FEATURES if c not in raw.columns]
    if missing:
        print(f'[{fname}] SKIPPED whole file: {len(missing)} feature columns missing, e.g. {missing[:5]}')
        for i in range(len(raw)):
            discard_log.append({'file': fname, 'row': i, 'source': infer_source(fname), 'Label': None,
                                'check': '1_format', 'reason': 'missing feature columns'})
        continue
    df = raw[FEATURES].apply(pd.to_numeric, errors='coerce').astype(np.float64)
    df['Label'] = raw['Label'].astype(str).str.strip().str.upper().map(label_lookup) if 'Label' in raw else np.nan
    df['source'] = raw['source'].astype(str).str.strip().str.lower() if 'source' in raw else infer_source(fname)
    df['source'] = df['source'].where(df['source'].isin(['llm', 'scripted']), infer_source(fname))
    df['source_row_id'] = pd.to_numeric(raw['source_row_id'], errors='coerce') if 'source_row_id' in raw else np.nan
    df['style'] = raw['style'].astype(str) if 'style' in raw else 'unspecified'
    df['generator'] = raw['generator'].astype(str) if 'generator' in raw else 'unspecified'
    df['file'] = fname; df['row'] = np.arange(len(df))

    bad_num = ~np.isfinite(df[FEATURES].to_numpy()).all(axis=1)
    bad_lab = df['Label'].isna().to_numpy()
    for i in np.where(bad_num | bad_lab)[0]:
        discard_log.append({'file': fname, 'row': int(i), 'source': df['source'].iat[i], 'Label': df['Label'].iat[i],
                            'check': '1_format', 'reason': 'non-numeric/NaN/Inf feature' if bad_num[i] else 'invalid Label'})
    print(f'[{fname}] {len(df):,} rows | format failures: {int((bad_num | bad_lab).sum()):,}')
    frames.append(df.loc[~(bad_num | bad_lab)])

V = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=FEATURES + META_COLS)
INPUT_ROWS = sum(len(fr) for fr in frames) + sum(1 for d in discard_log if d['check'] == '1_format')
print(f'\nRows read: {INPUT_ROWS:,} | passed check 1: {len(V):,}')
print(V.groupby(['source', 'Label']).size().rename('rows').to_string() if len(V) else 'no rows')

## 5 · Check (3a) non-negativity, then check (2) range clipping
Behavioural (mutable) features cannot be negative; such rows are **discarded** before clipping so they are not silently turned into zeros. All features are then clipped to the **training** `[min, max]` (or rows outside the range are discarded when `RANGE_MODE = 'discard'`).

In [ ]:
def log_discards(mask, check, reason):
    # reason: one string for all rows, or an array aligned with the current V
    global V
    mask = np.asarray(mask, dtype=bool)
    sub = V.loc[mask, ['file', 'row', 'source', 'Label']]
    reasons = [reason] * len(sub) if isinstance(reason, str) else list(np.asarray(reason, dtype=object)[mask])
    for (f, r, s, l), why in zip(sub.itertuples(index=False, name=None), reasons):
        discard_log.append({'file': f, 'row': int(r), 'source': s, 'Label': l, 'check': check, 'reason': why})
    V = V.loc[~mask].reset_index(drop=True)

neg = (V[MUTABLE].to_numpy() < 0).any(axis=1)
log_discards(neg, '3_consistency', 'negative value in a mutable feature')
print(f'Check 3a (non-negative mutable features): discarded {int(neg.sum()):,}')

lo = np.array([GBOUNDS[f]['min'] for f in FEATURES]); hi = np.array([GBOUNDS[f]['max'] for f in FEATURES])
Xv = V[FEATURES].to_numpy()
below, above = Xv < lo, Xv > hi
out_rows = (below | above).any(axis=1)
clip_report = {'rows_with_out_of_range_values': int(out_rows.sum()),
               'values_clipped_low': int(below.sum()), 'values_clipped_high': int(above.sum()),
               'most_clipped_features': pd.Series((below | above).sum(axis=0), index=FEATURES)
                                          .sort_values(ascending=False).head(10).to_dict()}
if RANGE_MODE == 'discard':
    log_discards(out_rows, '2_range', 'value outside training [min, max]')
    print(f'Check 2 (range, discard mode): discarded {int(out_rows.sum()):,}')
else:
    V[FEATURES] = np.clip(Xv, lo, hi)
    print(f'Check 2 (range, clip mode): {clip_report["rows_with_out_of_range_values"]:,} rows clipped '
          f'({clip_report["values_clipped_low"]:,} low / {clip_report["values_clipped_high"]:,} high values)')
print('Most clipped features:', {k: v for k, v in clip_report['most_clipped_features'].items() if v})
del Xv, below, above

## 6 · Check (3b) logical consistency rules
Rules are built from the columns that exist after preprocessing:
- `Max ≥ Mean ≥ Min` for every stat family (packet lengths, IATs, active/idle), and `Max Packet Length ≥ Min Packet Length`;
- at least one forward packet;
- IAT maxima / totals not longer than the flow duration;
- **duration vs rate consistency:** `Flow Packets/s`, `Fwd/Bwd Packets/s` and `Flow Bytes/s` within `RATE_TOLERANCE`× of the rate implied by counts / bytes ÷ duration;
- mean forward packet length consistent with total forward bytes ÷ forward packets.

Each rule is first **calibrated on the real training attack flows**. Only rules that ≥ 98 % of real flows satisfy are applied, so float rounding or CICFlowMeter quirks cannot discard valid variants. The calibration table is printed and saved.

In [ ]:
F = set(FEATURES)
EPS = 1e-6
def col(df, c): return df[c].to_numpy(dtype=np.float64)

rules = {}
for c in FEATURES:
    if c.endswith(' Max'):
        base = c[:-4]
        if base + ' Min' in F:
            rules[f'{base}: Max >= Min'] = (lambda b: lambda d: col(d, b + ' Max') + EPS >= col(d, b + ' Min'))(base)
        if base + ' Mean' in F and base + ' Min' in F:
            rules[f'{base}: Min <= Mean <= Max'] = (lambda b: lambda d:
                (col(d, b + ' Mean') + EPS >= col(d, b + ' Min')) & (col(d, b + ' Mean') <= col(d, b + ' Max') * (1 + 1e-5) + EPS))(base)
if {'Max Packet Length', 'Min Packet Length'} <= F:
    rules['Max Packet Length >= Min Packet Length'] = lambda d: col(d, 'Max Packet Length') + EPS >= col(d, 'Min Packet Length')
if {'Max Packet Length', 'Min Packet Length', 'Packet Length Mean'} <= F:
    rules['Min <= Packet Length Mean <= Max'] = lambda d: (col(d, 'Packet Length Mean') + EPS >= col(d, 'Min Packet Length')) & \
                                                         (col(d, 'Packet Length Mean') <= col(d, 'Max Packet Length') * (1 + 1e-5) + EPS)
if 'Total Fwd Packets' in F:
    rules['Total Fwd Packets >= 1'] = lambda d: col(d, 'Total Fwd Packets') >= 1
if 'Flow Duration' in F:
    for c in ['Flow IAT Max', 'Fwd IAT Total', 'Bwd IAT Total', 'Fwd IAT Max', 'Bwd IAT Max']:
        if c in F:
            rules[f'{c} <= Flow Duration'] = (lambda c: lambda d: col(d, c) <= col(d, 'Flow Duration') * (1 + 1e-4) + 1)(c)

def rate_rule(rate_col, numer_cols):
    def f(d):
        dur_s = col(d, 'Flow Duration') / DURATION_UNIT_PER_SECOND
        num = sum(col(d, c) for c in numer_cols)
        rate = col(d, rate_col)
        ok = np.ones(len(d), dtype=bool)
        pos = dur_s > 0
        implied = np.where(pos, num / np.where(pos, dur_s, 1), np.nan)
        both_zero = pos & (implied == 0) & (rate == 0)
        ratio = np.where(pos & (implied > 0), rate / np.where(implied > 0, implied, 1), np.nan)
        good = both_zero | ((ratio >= 1 / RATE_TOLERANCE) & (ratio <= RATE_TOLERANCE))
        ok[pos] = good[pos]
        return ok
    return f
if 'Flow Duration' in F:
    for rc, nc in [('Flow Packets/s', ['Total Fwd Packets', 'Total Backward Packets']),
                   ('Fwd Packets/s', ['Total Fwd Packets']), ('Bwd Packets/s', ['Total Backward Packets']),
                   ('Flow Bytes/s', ['Total Length of Fwd Packets', 'Total Length of Bwd Packets'])]:
        if rc in F and all(c in F for c in nc):
            rules[f'{rc} consistent with duration'] = rate_rule(rc, nc)
if {'Fwd Packet Length Mean', 'Total Length of Fwd Packets', 'Total Fwd Packets'} <= F:
    def fwd_mean_rule(d):
        n = col(d, 'Total Fwd Packets'); implied = col(d, 'Total Length of Fwd Packets') / np.where(n > 0, n, 1)
        m = col(d, 'Fwd Packet Length Mean')
        return np.abs(m - implied) <= np.maximum(1.0, 0.05 * np.maximum(m, implied))
    rules['Fwd Packet Length Mean = Fwd bytes / Fwd packets'] = fwd_mean_rule

real_attacks = train_raw[train_raw['Label'].isin(ATTACK_CLASSES)]
calib = []
for name, fn in rules.items():
    calib.append({'rule': name, 'real_attack_pass_rate': float(fn(real_attacks).mean())})
calib = pd.DataFrame(calib)
calib['applied'] = calib['real_attack_pass_rate'] >= RULE_MIN_REAL_PASS
calib.to_csv(os.path.join(DIRS['results'], '04_consistency_rule_calibration.csv'), index=False)
print(calib.to_string(index=False))

active_rules = [r for r in calib.loc[calib['applied'], 'rule']]
rule_fail_counts = {}
fail_any = np.zeros(len(V), dtype=bool); first_fail = np.array([''] * len(V), dtype=object)
for r in active_rules:
    ok = rules[r](V)
    rule_fail_counts[r] = int((~ok).sum())
    newly = ~ok & ~fail_any
    first_fail[newly] = r
    fail_any |= ~ok
log_discards(fail_any, '3_consistency', np.array(['rule failed: ' + x for x in first_fail], dtype=object))
print(f'\nCheck 3b (consistency): {len(active_rules)} rules applied, discarded {int(fail_any.sum()):,}')
print({k: v for k, v in rule_fail_counts.items() if v})

## 7 · Check (4) fixed-feature integrity
- **With `source_row_id`:** every fixed feature must equal the source flow's value, and the variant must keep the source flow's class.
- **Without it:** every fixed feature value must be one that real training flows of the **same class** take (e.g. the destination ports and flag values seen for that attack). Features with too many distinct values are range-checked within the class instead.

In [ ]:
train_fixed = train_raw[FIXED].to_numpy(dtype=np.float64)
fail4 = np.zeros(len(V), dtype=bool); reason4 = np.array([''] * len(V), dtype=object)

has_src = V['source_row_id'].notna().to_numpy()
if has_src.any():
    sid = V.loc[has_src, 'source_row_id'].astype(int).to_numpy()
    valid_sid = (sid >= 0) & (sid < len(train_raw))
    idx_has = np.where(has_src)[0]
    bad_id = idx_has[~valid_sid]
    fail4[bad_id] = True; reason4[bad_id] = 'source_row_id out of range'
    idx_ok, sid_ok = idx_has[valid_sid], sid[valid_sid]
    src_fixed = train_fixed[sid_ok]
    var_fixed = V.iloc[idx_ok][FIXED].to_numpy(dtype=np.float64)
    changed = ~np.isclose(var_fixed, src_fixed, rtol=FIXED_RTOL, atol=1e-6)
    label_mismatch = V.iloc[idx_ok]['Label'].to_numpy() != train_raw['Label'].to_numpy()[sid_ok]
    for k, i in enumerate(idx_ok):
        if label_mismatch[k]:
            fail4[i] = True; reason4[i] = 'class differs from source flow'
        elif changed[k].any():
            fail4[i] = True; reason4[i] = 'fixed feature changed: ' + ', '.join(np.array(FIXED)[changed[k]][:3])

allowed = {}
for c in ATTACK_CLASSES:
    sub = train_raw.loc[train_raw['Label'] == c, FIXED]
    allowed[c] = {}
    for f in FIXED:
        vals = sub[f].to_numpy(dtype=np.float64)
        u = np.unique(np.round(vals, 4))
        allowed[c][f] = ('set', u) if len(u) <= 2000 else ('range', (vals.min(), vals.max()))
no_src = np.where(~has_src)[0]
for c in ATTACK_CLASSES:
    idx = no_src[V['Label'].to_numpy()[no_src] == c]
    if len(idx) == 0:
        continue
    for f in FIXED:
        kind, a = allowed[c][f]
        v = V[f].to_numpy(dtype=np.float64)[idx]
        bad = ~np.isin(np.round(v, 4), a) if kind == 'set' else (v < a[0]) | (v > a[1])
        newly = idx[bad & ~fail4[idx]]
        fail4[newly] = True; reason4[newly] = f'fixed feature value never seen for {c}: {f}'

log_discards(fail4, '4_fixed_integrity', reason4)
print(f'Check 4 (fixed-feature integrity): {int(has_src.sum()):,} rows checked against their source flow, '
      f'{int((~has_src).sum()):,} against per-class allowed values | discarded {int(fail4.sum()):,}')

## 8 · Check (5) distribution similarity + copy / duplicate removal
1. **Copies** — a variant identical to a real training flow is not a variant (and would leak), so it is discarded. Exact duplicates inside the variant set are discarded too.
2. **Similarity sanity check** (report only, no discards) — per source and class: mean and std of every mutable feature for real vs. variant flows, standardised mean difference (SMD) and the two-sample KS statistic, plus comparison plots. Variants should be *different but related*: a median KS near 0 means the variants are almost copies; a median |SMD| ≫ 3 means they no longer resemble the attack.

In [ ]:
def row_hash(frame):
    return pd.util.hash_pandas_object(frame[FEATURES].astype(np.float64).round(3), index=False).to_numpy()

train_hashes = set(row_hash(train_raw))
h = row_hash(V)
is_copy = np.array([x in train_hashes for x in h])
log_discards(is_copy, '5_similarity', 'identical to a real training flow')
h = row_hash(V)
is_dup = pd.Series(h).duplicated(keep='first').to_numpy()
log_discards(is_dup, '5_similarity', 'duplicate of another variant')
print(f'Check 5 (copies/duplicates): discarded {int(is_copy.sum()):,} copies of real flows and {int(is_dup.sum()):,} duplicates')

rng = np.random.default_rng(DATA_SEED)
sim_rows = []
for src in sorted(V['source'].unique()):
    for c in ATTACK_CLASSES:
        var = V[(V['source'] == src) & (V['Label'] == c)]
        org = train_raw[train_raw['Label'] == c]
        if len(var) < 5 or len(org) < 5:
            continue
        o_s = org.sample(min(len(org), KS_SUBSAMPLE), random_state=DATA_SEED)
        v_s = var.sample(min(len(var), KS_SUBSAMPLE), random_state=DATA_SEED)
        for f in MUTABLE:
            mo, so = float(org[f].mean()), float(org[f].std())
            mv, sv = float(var[f].mean()), float(var[f].std())
            pooled = np.sqrt((so ** 2 + sv ** 2) / 2)
            smd = (mv - mo) / pooled if pooled > 0 else 0.0
            ks = ks_2samp(o_s[f].to_numpy(), v_s[f].to_numpy()).statistic
            sim_rows.append({'source': src, 'class': c, 'feature': f, 'orig_mean': mo, 'var_mean': mv,
                             'orig_std': so, 'var_std': sv, 'smd': smd, 'ks': ks})
sim = pd.DataFrame(sim_rows)
sim.to_csv(os.path.join(DIRS['results'], '04_similarity_per_feature.csv'), index=False)
if len(sim):
    sim_summary = sim.groupby(['source', 'class']).agg(median_abs_smd=('smd', lambda s: float(np.median(np.abs(s)))),
                                                         median_ks=('ks', 'median'), max_ks=('ks', 'max'))
    sim_summary['verdict'] = np.where(sim_summary['median_ks'] < 0.02, 'WARNING: almost copies of real flows',
                              np.where(sim_summary['median_abs_smd'] > 3, 'WARNING: far from real attack profile',
                                       'different but related'))
    sim_summary.to_csv(os.path.join(DIRS['results'], '04_similarity_summary.csv'))
    print(sim_summary.to_string())

In [ ]:
key = [f for f in ['Flow Duration', 'Total Fwd Packets', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean',
                   'Packet Length Mean'] if f in MUTABLE] or MUTABLE[:6]
sources = sorted(V['source'].unique())
fig, axes = plt.subplots(len(ATTACK_CLASSES), len(key), figsize=(3.1 * len(key), 2.6 * len(ATTACK_CLASSES)), squeeze=False)
style = {'real': '#4a4a4a', 'llm': '#2f6db5', 'scripted': '#d1495b'}
for i, c in enumerate(ATTACK_CLASSES):
    for j, f in enumerate(key):
        ax = axes[i, j]
        series = {'real': train_raw.loc[train_raw['Label'] == c, f]}
        for s in sources:
            series[s] = V.loc[(V['source'] == s) & (V['Label'] == c), f]
        allv = np.log1p(np.concatenate([v.to_numpy() for v in series.values() if len(v)]))
        bins = np.linspace(allv.min(), allv.max() + 1e-9, 40)
        for name, v in series.items():
            if len(v):
                ax.hist(np.log1p(v.to_numpy()), bins=bins, density=True, histtype='step', lw=1.5,
                        color=style.get(name, 'k'), label=f'{name} (n={len(v):,})')
        if i == 0: ax.set_title(f'log1p({f})', fontsize=8)
        if j == 0: ax.set_ylabel(c, fontsize=9)
        ax.tick_params(labelsize=7); ax.spines[['top', 'right']].set_visible(False)
        if j == len(key) - 1: ax.legend(fontsize=6, frameon=False)
plt.suptitle('Check 5 — real training attacks vs validated variants (density, log1p scale)')
plt.tight_layout()
plt.savefig(os.path.join(DIRS['results'], '04_similarity_distributions.png'), dpi=140)
plt.show()

## 9 · Validation summary — discard counts and rates

In [ ]:
dl = pd.DataFrame(discard_log, columns=['file', 'row', 'source', 'Label', 'check', 'reason'])
dl.to_csv(os.path.join(DIRS['results'], '04_discarded_variants.csv'), index=False)

per_check = dl.groupby(['source', 'check']).size().unstack(fill_value=0) if len(dl) else pd.DataFrame()
kept = V.groupby('source').size()
in_by_src = (dl.groupby('source').size() if len(dl) else pd.Series(dtype=int)).add(kept, fill_value=0)
summary = pd.DataFrame({'input_rows': in_by_src, 'kept': kept}).fillna(0).astype(int)
summary['discarded'] = summary['input_rows'] - summary['kept']
summary['discard_rate_%'] = (100 * summary['discarded'] / summary['input_rows'].clip(lower=1)).round(2)
summary = summary.join(per_check).fillna(0)
summary.to_csv(os.path.join(DIRS['results'], '04_validation_summary.csv'))
print('Discards per source and check:'); print(summary.to_string())
print('\nKept rows per source and class:')
kept_tab = V.groupby(['source', 'Label']).size().unstack(fill_value=0)
print(kept_tab.to_string())
print('\nTop discard reasons:')
print(dl['reason'].value_counts().head(10).to_string() if len(dl) else 'none')

TOTAL_IN = INPUT_ROWS
print(f'\nTOTAL: {TOTAL_IN:,} input rows -> {len(V):,} kept | discarded {TOTAL_IN - len(V):,} '
      f'({100 * (TOTAL_IN - len(V)) / max(TOTAL_IN, 1):.2f} %)')
validation_report = {'input_rows': int(TOTAL_IN), 'kept_rows': int(len(V)), 'discarded_rows': int(TOTAL_IN - len(V)),
                     'discard_rate': (TOTAL_IN - len(V)) / max(TOTAL_IN, 1), 'range_mode': RANGE_MODE,
                     'clip_report': clip_report, 'rules_applied': active_rules, 'rule_fail_counts': rule_fail_counts,
                     'per_source': summary.reset_index().to_dict(orient='records'),
                     'kept_per_source_class': kept_tab.to_dict(), 'files': [os.path.basename(f) for f in variant_files]}
with open(os.path.join(DIRS['results'], '04_validation_report.json'), 'w') as fh:
    json.dump(validation_report, fh, indent=2, default=str)

## 10 · Part C — Split: `variant_train_pool` / `variant_test_heldout` (LLM) and the scripted control set
The split is stratified by class. When `source_row_id` is available it is **group-aware**: all variants derived from the same real flow go to the same side, so the held-out set is genuinely unseen. Scripted variants are used only as the E3 control test set.

In [ ]:
V['label_id'] = V['Label'].map(LABEL_TO_ID).astype(np.int64)
V['variant_id'] = np.arange(len(V))
llm = V[V['source'] == 'llm'].reset_index(drop=True)
scr = V[V['source'] == 'scripted'].reset_index(drop=True)

pool_parts, held_parts = [], []
for c in ATTACK_CLASSES:
    sub = llm[llm['Label'] == c]
    if len(sub) == 0:
        continue
    if len(sub) < 4:
        held_parts.append(sub); continue
    groups = sub['source_row_id'].fillna(-1 - sub['variant_id']).to_numpy()   # rows without a source id are their own group
    if len(np.unique(groups)) >= 2:
        gss = GroupShuffleSplit(n_splits=1, test_size=HELDOUT_FRACTION, random_state=DATA_SEED)
        tr_i, te_i = next(gss.split(sub, groups=groups))
    else:
        tr_i, te_i = train_test_split(np.arange(len(sub)), test_size=HELDOUT_FRACTION, random_state=DATA_SEED)
    pool_parts.append(sub.iloc[tr_i]); held_parts.append(sub.iloc[te_i])

variant_train_pool = pd.concat(pool_parts, ignore_index=True) if pool_parts else llm.iloc[:0]
variant_test_heldout = pd.concat(held_parts, ignore_index=True) if held_parts else llm.iloc[:0]
if len(variant_train_pool) and variant_train_pool['source_row_id'].notna().any():
    shared = set(variant_train_pool['source_row_id'].dropna()) & set(variant_test_heldout['source_row_id'].dropna())
    print('source flows shared between pool and held-out:', len(shared))
    assert len(shared) == 0

keep_cols = FEATURES + ['Label', 'label_id', 'source', 'style', 'generator', 'source_row_id', 'file', 'variant_id']
variant_train_pool[keep_cols].to_parquet(os.path.join(DIRS['variants'], 'variant_train_pool.parquet'), index=False)
variant_test_heldout[keep_cols].to_parquet(os.path.join(DIRS['variants'], 'variant_test_heldout.parquet'), index=False)
scr[keep_cols].to_parquet(os.path.join(DIRS['variants'], 'scripted_control.parquet'), index=False)
V[keep_cols].to_parquet(os.path.join(DIRS['validated'], 'all_validated_variants.parquet'), index=False)

split_tab = pd.DataFrame({'variant_train_pool': variant_train_pool['Label'].value_counts(),
                          'variant_test_heldout': variant_test_heldout['Label'].value_counts(),
                          'scripted_control (E3)': scr['Label'].value_counts()}).reindex(ATTACK_CLASSES).fillna(0).astype(int)
split_tab.loc['TOTAL'] = split_tab.sum()
split_tab.to_csv(os.path.join(DIRS['results'], '04_variant_split.csv'))
print(split_tab)
if len(variant_test_heldout) == 0:
    raise ValueError('No validated LLM variants available for the held-out set (E2 cannot run).')

## 11 · Part D — Evaluate the baseline models (E1 reference, E2 LLM variants, E3 scripted control)
For every model × seed the saved E1 model is loaded and evaluated on:
- the **attack flows of the original test set** (reference recall, recomputed here so all numbers come from one code path);
- `variant_test_heldout` (**E2**);
- `scripted_control` (**E3**, skipped if empty).

Variants are scaled with the **same scaler** fitted in notebook 01.

In [ ]:
X_test = np.load(os.path.join(DIRS['processed'], 'X_test.npy')); y_test = np.load(os.path.join(DIRS['processed'], 'y_test.npy'))
att_mask = y_test != BENIGN_ID
EVAL_SETS = {'original': (X_test[att_mask], y_test[att_mask])}
def scale(frame):
    return scaler.transform(frame[FEATURES].to_numpy(dtype=np.float32)).astype(np.float32)
EVAL_SETS['E2_llm'] = (scale(variant_test_heldout), variant_test_heldout['label_id'].to_numpy())
if len(scr):
    EVAL_SETS['E3_scripted'] = (scale(scr), scr['label_id'].to_numpy())
for k, (X, y) in EVAL_SETS.items():
    print(f'{k:<12} {X.shape[0]:>8,} attack flows | per class:', dict(zip(ATTACK_CLASSES, [int((y == LABEL_TO_ID[c]).sum()) for c in ATTACK_CLASSES])))

def macro_attack_recall(m):
    vals = [m[f'recall_{c}'] for c in ATTACK_CLASSES if m.get(f'n_{c}', 0) > 0]
    return float(np.mean(vals)) if vals else float('nan')

records, cms = [], {}
for name in MODEL_NAMES:
    for seed in MODEL_SEEDS:
        path = model_path(DIRS['baseline_models'], name, seed)
        if not os.path.exists(path):
            print('missing model, run notebook 02 first:', path); continue
        model = load_model(name, path, device=DEVICE)
        for set_name, (X, y) in EVAL_SETS.items():
            pred = predict(name, model, X, device=DEVICE)
            m = attack_set_metrics(y, pred, CLASS_NAMES, BENIGN_ID)
            m.update({'model': name, 'seed': seed, 'set': set_name, 'recall_macro_attack': macro_attack_recall(m)})
            records.append(m)
            cm = np.zeros((len(CLASS_NAMES), len(CLASS_NAMES)), dtype=np.int64)
            np.add.at(cm, (y, pred), 1)
            cms.setdefault((name, set_name), []).append(cm)
        del model; gc.collect()
        print(f'evaluated {name} seed {seed}')

res = pd.DataFrame(records)
res.to_csv(os.path.join(DIRS['results'], '04_eval_runs_all_seeds.csv'), index=False)
print(res[['model', 'seed', 'set', 'recall_overall', 'recall_macro_attack', 'evasion_rate']].round(4).to_string(index=False))

## 12 · Part E — Recall Drop and Evasion Rate (mean ± std over seeds)
- **Recall Drop** (per class) = recall on original test attacks − recall on variants, computed **per seed** (same model instance), then averaged. `ALL` uses the class-balanced macro attack recall so different class mixes between sets do not distort the comparison.
- **Evasion Rate** = share of attack variants predicted `Benign`.

In [ ]:
long = []
for _, r in res.iterrows():
    long.append({'model': r['model'], 'seed': r['seed'], 'set': r['set'], 'class': 'ALL',
                 'recall': r['recall_macro_attack'], 'evasion': r['evasion_rate'], 'n': r['n_samples']})
    for c in ATTACK_CLASSES:
        if r.get(f'n_{c}', 0) > 0:
            long.append({'model': r['model'], 'seed': r['seed'], 'set': r['set'], 'class': c,
                         'recall': r[f'recall_{c}'], 'evasion': r[f'evasion_{c}'], 'n': r[f'n_{c}']})
long = pd.DataFrame(long)
orig = long[long['set'] == 'original'][['model', 'seed', 'class', 'recall', 'evasion']] \
    .rename(columns={'recall': 'orig_recall', 'evasion': 'orig_evasion'})
cmp_ = long[long['set'] != 'original'].merge(orig, on=['model', 'seed', 'class'], how='left')
cmp_['recall_drop'] = cmp_['orig_recall'] - cmp_['recall']
cmp_['evasion_increase'] = cmp_['evasion'] - cmp_['orig_evasion']
cmp_.to_csv(os.path.join(DIRS['results'], '04_recall_drop_evasion_all_seeds.csv'), index=False)

agg = cmp_.groupby(['set', 'model', 'class']).agg(
    orig_recall_mean=('orig_recall', 'mean'), variant_recall_mean=('recall', 'mean'), variant_recall_std=('recall', 'std'),
    recall_drop_mean=('recall_drop', 'mean'), recall_drop_std=('recall_drop', 'std'),
    evasion_mean=('evasion', 'mean'), evasion_std=('evasion', 'std'), n=('n', 'first')).reset_index()
agg = agg.fillna({'variant_recall_std': 0, 'recall_drop_std': 0, 'evasion_std': 0})
agg.to_csv(os.path.join(DIRS['results'], '04_robustness_summary.csv'), index=False)

def pm(m, s): return f'{m:.4f} ± {s:.4f}'
pretty = agg.assign(**{'Original recall': agg['orig_recall_mean'].map('{:.4f}'.format),
                       'Variant recall': [pm(a, b) for a, b in zip(agg['variant_recall_mean'], agg['variant_recall_std'])],
                       'Recall drop': [pm(a, b) for a, b in zip(agg['recall_drop_mean'], agg['recall_drop_std'])],
                       'Evasion rate': [pm(a, b) for a, b in zip(agg['evasion_mean'], agg['evasion_std'])]}) \
    [['set', 'model', 'class', 'n', 'Original recall', 'Variant recall', 'Recall drop', 'Evasion rate']]
pretty['model'] = pd.Categorical(pretty['model'], MODEL_NAMES, ordered=True)
pretty = pretty.sort_values(['set', 'model', 'class']).reset_index(drop=True)
pretty.to_csv(os.path.join(DIRS['results'], '04_robustness_summary_pretty.csv'), index=False)
for s in pretty['set'].unique():
    print(f'\n===== {s} =====')
    print(pretty[pretty['set'] == s].drop(columns='set').to_string(index=False))

if 'E3_scripted' in set(cmp_['set']):
    e = agg[agg['class'] == 'ALL'].pivot(index='model', columns='set', values=['recall_drop_mean', 'evasion_mean'])
    e.columns = [f'{a}_{b}' for a, b in e.columns]
    e['LLM_minus_scripted_drop'] = e['recall_drop_mean_E2_llm'] - e['recall_drop_mean_E3_scripted']
    e.to_csv(os.path.join(DIRS['results'], '04_E2_vs_E3.csv'))
    print('\nE2 (LLM) vs E3 (scripted noise), class-balanced:'); print(e.round(4).to_string())

In [ ]:
if V['style'].nunique() > 1:
    rows = []
    for name in MODEL_NAMES:
        for seed in MODEL_SEEDS:
            path = model_path(DIRS['baseline_models'], name, seed)
            if not os.path.exists(path): continue
            model = load_model(name, path, device=DEVICE)
            pred = predict(name, model, EVAL_SETS['E2_llm'][0], device=DEVICE)
            for st in variant_test_heldout['style'].unique():
                mk = (variant_test_heldout['style'] == st).to_numpy()
                rows.append({'model': name, 'seed': seed, 'style': st, 'n': int(mk.sum()),
                             'recall': float((pred[mk] == EVAL_SETS['E2_llm'][1][mk]).mean()),
                             'evasion': float((pred[mk] == BENIGN_ID).mean())})
            del model
    by_style = pd.DataFrame(rows).groupby(['model', 'style']).agg(n=('n', 'first'), recall=('recall', 'mean'),
                                                                   evasion=('evasion', 'mean')).round(4)
    by_style.to_csv(os.path.join(DIRS['results'], '04_E2_by_style.csv'))
    print('E2 by variation style (mean over seeds):'); print(by_style.to_string())
else:
    print('Only one style tag present -> per-style breakdown skipped.')

## 13 · Charts

In [ ]:
colors = {'original': '#4a4a4a', 'E2_llm': '#2f6db5', 'E3_scripted': '#d1495b'}
sets_present = [s for s in ['original', 'E2_llm', 'E3_scripted'] if s in set(long['set'])]

# (1) class-balanced recall: original vs variants, per model
fig, ax = plt.subplots(figsize=(9, 4.5))
x = np.arange(len(MODEL_NAMES)); w = 0.8 / len(sets_present)
for k, s in enumerate(sets_present):
    sub = long[(long['set'] == s) & (long['class'] == 'ALL')].groupby('model')['recall']
    mu = sub.mean().reindex(MODEL_NAMES); sd = sub.std().reindex(MODEL_NAMES).fillna(0)
    ax.bar(x + k * w - 0.4 + w / 2, mu, w, yerr=sd, capsize=3, color=colors[s], label=s)
ax.set_xticks(x); ax.set_xticklabels(MODEL_NAMES); ax.set_ylim(0, 1.05)
ax.set_ylabel('class-balanced attack recall'); ax.set_title('Recall on original test attacks vs variants (mean ± std, 3 seeds)')
ax.legend(frameon=False); ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout(); plt.savefig(os.path.join(DIRS['results'], '04_recall_original_vs_variants.png'), dpi=150); plt.show()

# (2) evasion-rate heatmaps
var_sets = [s for s in ['E2_llm', 'E3_scripted'] if s in set(agg['set'])]
fig, axes = plt.subplots(1, len(var_sets), figsize=(5.5 * len(var_sets), 3.8), squeeze=False)
for ax, s in zip(axes[0], var_sets):
    piv = agg[(agg['set'] == s) & (agg['class'] != 'ALL')].pivot(index='model', columns='class', values='evasion_mean') \
        .reindex(index=MODEL_NAMES, columns=[c for c in ATTACK_CLASSES])
    im = ax.imshow(piv.to_numpy(dtype=float), cmap='Reds', vmin=0, vmax=max(0.05, float(np.nanmax(piv.to_numpy(dtype=float)))))
    for i in range(piv.shape[0]):
        for j in range(piv.shape[1]):
            v = piv.iat[i, j]
            ax.text(j, i, 'n/a' if pd.isna(v) else f'{v:.3f}', ha='center', va='center', fontsize=9)
    ax.set_xticks(range(piv.shape[1])); ax.set_xticklabels(piv.columns, rotation=20)
    ax.set_yticks(range(piv.shape[0])); ax.set_yticklabels(piv.index); ax.set_title(f'Evasion rate — {s}')
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout(); plt.savefig(os.path.join(DIRS['results'], '04_evasion_rate_heatmap.png'), dpi=150); plt.show()

# (3) recall drop per class (E2)
fig, ax = plt.subplots(figsize=(9, 4.5))
sub = agg[(agg['set'] == 'E2_llm') & (agg['class'] != 'ALL')]
w = 0.8 / len(MODEL_NAMES); xc = np.arange(len(ATTACK_CLASSES))
palette = ['#2f6db5', '#d1495b', '#edae49', '#00798c']
for k, name in enumerate(MODEL_NAMES):
    s2 = sub[sub['model'] == name].set_index('class').reindex(ATTACK_CLASSES)
    ax.bar(xc + k * w - 0.4 + w / 2, s2['recall_drop_mean'], w, yerr=s2['recall_drop_std'], capsize=3,
           color=palette[k], label=name)
ax.axhline(0, color='k', lw=0.8); ax.set_xticks(xc); ax.set_xticklabels(ATTACK_CLASSES)
ax.set_ylabel('recall drop (original − variant)'); ax.set_title('E2 — recall drop per attack type')
ax.legend(frameon=False); ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout(); plt.savefig(os.path.join(DIRS['results'], '04_E2_recall_drop_per_class.png'), dpi=150); plt.show()

# (4) where do the E2 variants go? pooled confusion (attack rows only)
fig, axes = plt.subplots(1, len(MODEL_NAMES), figsize=(4.4 * len(MODEL_NAMES), 3.4), squeeze=False)
att_ids = [LABEL_TO_ID[c] for c in ATTACK_CLASSES]
for ax, name in zip(axes[0], MODEL_NAMES):
    if (name, 'E2_llm') not in cms: ax.axis('off'); continue
    cm = np.sum(cms[(name, 'E2_llm')], axis=0)[att_ids]
    cmn = cm / cm.sum(axis=1, keepdims=True).clip(min=1)
    ax.imshow(cmn, cmap='Blues', vmin=0, vmax=1)
    for i in range(cmn.shape[0]):
        for j in range(cmn.shape[1]):
            ax.text(j, i, f'{cmn[i, j]:.2f}', ha='center', va='center', fontsize=8, color='white' if cmn[i, j] > 0.6 else 'black')
    ax.set_xticks(range(len(CLASS_NAMES))); ax.set_xticklabels(CLASS_NAMES, rotation=30, fontsize=7)
    ax.set_yticks(range(len(ATTACK_CLASSES))); ax.set_yticklabels(ATTACK_CLASSES, fontsize=7); ax.set_title(name)
plt.suptitle('E2 — predictions on held-out LLM variants (3 seeds pooled, row-normalised)')
plt.tight_layout(); plt.savefig(os.path.join(DIRS['results'], '04_E2_confusion.png'), dpi=150); plt.show()
with open(os.path.join(DIRS['results'], '04_confusion_matrices.json'), 'w') as fh:
    json.dump({f'{k[0]}|{k[1]}': np.sum(v, axis=0).tolist() for k, v in cms.items()}, fh)

## 14 · Summary

In [ ]:
print('=' * 72); print('STAGE 4 / EXPERIMENTS E2 & E3 COMPLETE'); print('=' * 72)
print(f'Validation: {validation_report["input_rows"]:,} rows in, {validation_report["kept_rows"]:,} kept, '
      f'discard rate {100 * validation_report["discard_rate"]:.2f} %')
print(split_tab.to_string())
headline = agg[agg['class'] == 'ALL'][['set', 'model', 'orig_recall_mean', 'variant_recall_mean', 'recall_drop_mean', 'evasion_mean']]
print('\nHeadline (class-balanced, mean over seeds):'); print(headline.round(4).to_string(index=False))
print('\nFiles for notebook 05: data/variants/variant_train_pool.parquet, data/variants/variant_test_heldout.parquet')
print('Next: run 05_adversarial_retraining.ipynb')